# Camada Silver — `redes`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Conversão de Tipos (`id_aluno`, `interacoes`, `seguidores`, `seguindo`), Validação (`>= 0`), Detecção de Outliers (IQR e Z-Score) e Remoção de Duplicatas.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("redes", BRONZE_DIR)
rows_in = len(df)

df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
for col in ["interacoes", "seguidores", "seguindo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0).astype("Int64")

df["flag_interacoes_outlier_iqr"], iqr_int = detect_outliers_iqr(df["interacoes"].astype(float))
df["flag_interacoes_outlier_zscore"], z_int = detect_outliers_zscore(df["interacoes"].astype(float))

dups_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

log = {
    "dataset": "redes",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "interacoes_outliers_iqr": iqr_int,
    "interacoes_outliers_zscore": z_int,
}
save_silver_outputs(df, log, "redes", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)